# 03 Integrity audit (Phase 3)
Read-only. Findings: `results/audit/findings.csv`; report: `reports/dataset_audit_report.md`. Nothing was repaired or dropped.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
R = ROOT / "results/audit"
M = ROOT / "data/manifests"
pd.set_option("display.max_rows", 80); pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
print("project root:", ROOT.name)

project root: research_paper


In [2]:
F = pd.read_csv(R / 'findings.csv')
F.groupby(['category','severity']).size().unstack(fill_value=0)

severity,high,low,medium
category,,,
COMPLETENESS,7,0,0
CONSTANT,0,17,0
DUPLICATES,0,0,8
DUPLICATE_COLUMN,0,11,0
EXPORT_OVERFLOW,4,0,0
FORMAT,1,1,1
IMBALANCE,0,0,34
INVALID_VALUE,18,0,0
LABEL_CONFLICT,4,0,0


In [3]:
F[F.severity.isin(['critical','high'])][['finding_id','dataset_family','variant','scope','column','rule','count','denominator','rate']]

,finding_id,dataset_family,variant,scope,column,rule,count,denominator,rate
1,FORMAT-002,CSE-CIC-IDS-2018,original,data/raw_original/CSE-CIC-IDS-2018/Thuesday-20...,NaN,mixed CRLF and LF line terminators,577787,7948749.0,7.268905e-02
2,COMPLETENESS-001,CSE-CIC-IDS-2018,original,data/raw_original/CSE-CIC-IDS-2018/Friday-02-0...,NaN,"data rows == 1,048,575 (Excel row limit 2^20 m...",1048575,NaN,NaN
3,COMPLETENESS-002,CSE-CIC-IDS-2018,original,data/raw_original/CSE-CIC-IDS-2018/Friday-16-0...,NaN,"data rows == 1,048,575 (Excel row limit 2^20 m...",1048575,NaN,NaN
4,COMPLETENESS-003,CSE-CIC-IDS-2018,original,data/raw_original/CSE-CIC-IDS-2018/Friday-23-0...,NaN,"data rows == 1,048,575 (Excel row limit 2^20 m...",1048575,NaN,NaN
5,COMPLETENESS-004,CSE-CIC-IDS-2018,original,data/raw_original/CSE-CIC-IDS-2018/Thursday-15...,NaN,"data rows == 1,048,575 (Excel row limit 2^20 m...",1048575,NaN,NaN
6,COMPLETENESS-005,CSE-CIC-IDS-2018,original,data/raw_original/CSE-CIC-IDS-2018/Thursday-22...,NaN,"data rows == 1,048,575 (Excel row limit 2^20 m...",1048575,NaN,NaN
7,COMPLETENESS-006,CSE-CIC-IDS-2018,original,data/raw_original/CSE-CIC-IDS-2018/Wednesday-1...,NaN,"data rows == 1,048,575 (Excel row limit 2^20 m...",1048575,NaN,NaN
8,COMPLETENESS-007,CSE-CIC-IDS-2018,original,data/raw_original/CSE-CIC-IDS-2018/Wednesday-2...,NaN,"data rows == 1,048,575 (Excel row limit 2^20 m...",1048575,NaN,NaN
9,MISSING-001,CIC-IDS-2017,original,dataset,all,rows where every field is empty,288602,3119345.0,9.252006e-02
20,INVALID_VALUE-001,CIC-IDS-2017,original,dataset,Flow Duration,negative value in a column that cannot be nega...,115,2830743.0,4.063000e-05


In [4]:
D = pd.read_csv(R / 'duplicates_by_file.csv').query("scope == 'dataset'")
D = D.assign(feature_dup_share=D.n_feature_duplicate_rows / D.n_data_rows)
D.set_index(['dataset_family','variant']).T

dataset_family,CIC-IDS-2017,CSE-CIC-IDS-2018,CIC-IDS-2017,CSE-CIC-IDS-2018
variant,improved,improved,original,original
scope,dataset,dataset,dataset,dataset
file,NaN,NaN,NaN,NaN
n_rows,2099976,63195145,3119345,16233002
n_all_empty_rows,0,0,288602,0
n_embedded_header_rows,0,0,0,59
n_data_rows,2099976,63195145,2830743,16232943
n_exact_duplicate_rows_excl_empty,0,0,203,433195
n_feature_duplicate_rows,263004,17994469,308848,4265470
n_feature_duplicate_rows_incl_nondata,263004,17994469,597449,4265528


In [5]:
T = pd.read_csv(R / 'timestamp_quality.csv')
T[['file','n_rows','n_parse_failures','min_ts','max_ts','n_duplicate_ts','n_backward_steps_gt_60s','max_gap_sorted_s','n_id_ne_row_order']]

,file,n_rows,n_parse_failures,min_ts,max_ts,n_duplicate_ts,n_backward_steps_gt_60s,max_gap_sorted_s,n_id_ne_row_order
0,data/raw_improved/CIC-IDS-2017/friday.csv,547557,0,2017-07-07 11:59:50.315195,2017-07-07 20:02:40.977798,8306,223676,4.107229e+01,0
1,data/raw_improved/CIC-IDS-2017/monday.csv,371624,0,2017-07-03 11:55:58.598308,2017-07-03 20:01:34.352593,347,166250,2.373271e+01,0
2,data/raw_improved/CIC-IDS-2017/thursday.csv,362076,0,2017-07-06 11:59:00.490140,2017-07-06 20:04:36.513760,408,158136,4.092188e+01,0
3,data/raw_improved/CIC-IDS-2017/tuesday.csv,322078,0,2017-07-04 11:53:44.398274,2017-07-04 20:00:30.041663,211,144797,3.632294e+01,0
4,data/raw_improved/CIC-IDS-2017/wednesday.csv,496641,0,2017-07-05 11:42:42.084372,2017-07-05 20:10:14.557249,13693,150689,5.933770e+01,0
5,data/raw_improved/CSE-CIC-IDS-2018/Friday-02-0...,6311371,0,2018-03-02 12:46:22.317145,2018-03-03 00:39:53.894472,1157,2754260,3.628526e+01,0
6,data/raw_improved/CSE-CIC-IDS-2018/Friday-16-0...,7390266,0,2018-02-16 12:26:55.181877,2018-02-16 22:15:24.854148,1042,2394753,1.303488e+02,0
7,data/raw_improved/CSE-CIC-IDS-2018/Friday-23-0...,5976481,0,2018-02-21 12:33:55.396864,2018-02-23 23:46:49.183594,977,2598606,8.599121e+04,0
8,data/raw_improved/CSE-CIC-IDS-2018/Thursday-01...,6551401,0,2018-03-01 12:15:28.926174,2018-03-01 23:40:14.412141,1340,2845692,2.629029e+01,0
9,data/raw_improved/CSE-CIC-IDS-2018/Thursday-15...,5410102,0,2018-02-15 12:23:15.264383,2018-02-16 01:14:55.132078,667,2312945,2.433092e+01,0


In [6]:
pd.read_csv(R / 'label_window_consistency.csv')

,dataset_family,file,label,attempted_category,n_flows,n_with_documented_rule,n_timestamp_inside_rule_window,n_outside,first_ts,last_ts
0,CIC-IDS-2017,friday.csv,Botnet,-1,736,736,736,0,2017-07-07 13:04:13.990571,2017-07-07 14:02:02.808963
1,CIC-IDS-2017,friday.csv,Botnet - Attempted,1,4067,4067,4067,0,2017-07-07 14:03:14.356593,2017-07-07 20:01:12.727507
2,CIC-IDS-2017,friday.csv,DDoS,-1,95144,95144,95144,0,2017-07-07 18:56:31.796937,2017-07-07 19:16:10.645771
3,CIC-IDS-2017,friday.csv,Portscan,-1,159066,159066,159066,0,2017-07-07 16:55:32.117090,2017-07-07 18:23:53.256235
4,CIC-IDS-2017,thursday.csv,Infiltration,-1,36,36,36,0,2017-07-06 17:19:02.547210,2017-07-06 18:44:24.375292
...,...,...,...,...,...,...,...,...,...,...
80,CSE-CIC-IDS-2018,Wednesday-21-02-2018.csv,DDoS-LOIC-UDP - Attempted,6,171,171,171,0,2018-02-21 14:08:51.251350,2018-02-21 14:43:02.204173
81,CSE-CIC-IDS-2018,Wednesday-28-02-2018.csv,Infiltration - Communication Victim Attacker,-1,43,43,43,0,2018-02-28 14:45:40.706215,2018-02-28 18:39:59.703008
82,CSE-CIC-IDS-2018,Wednesday-28-02-2018.csv,Infiltration - Dropbox Download,-1,46,46,46,0,2018-02-28 14:33:24.627743,2018-02-28 17:43:26.452368
83,CSE-CIC-IDS-2018,Wednesday-28-02-2018.csv,Infiltration - Dropbox Download - Attempted,4,15,15,15,0,2018-02-28 14:33:25.142961,2018-02-28 17:42:54.554905


In [7]:
# independent recount of the all-empty rows in the original 2017 web-attack file (plain bytes)
p = ROOT / 'data/raw_original/CIC-IDS-2017/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv'
n_empty = sum(1 for line in p.open('rb') if line.strip(b'\r\n') and not line.strip(b',\r\n'))
print('comma-only lines:', n_empty)

comma-only lines: 288602
